# Predicting Time and Fares of taxi runs using Spark

- Libero Biagi 20250349
- Marcell Hajdù 20250377
- Jenny Cubelo 20250431
- Emily Garrone 20250430

In this project for the Big Data Analytics course, our objective is to use the [NYC Yellow Taxi Trip Data](https://www.kaggle.com/datasets/elemento/nyc-yellow-taxi-trip-data?select=yellow_tripdata_2016-01.csv) to try to predict the duration of rides and the fare that will be applied to them. We will use very basic features to simulate a possible model that can be embedded into an app used by passengers to book trips. For this, we decided to implement some possible lightweight algorithms and a streaming process, but let's take a look at the notebook.

### Imports

In [1]:
import os
import random

# --- DATA PROCESSING & NUMERICAL COMPUTING ---
import numpy as np
import pandas as pd

# --- DEEP LEARNING (TORCH) ---
import torch
import torch.nn as nn
import torch.nn.functional as Functional
from torch.utils.data import DataLoader, TensorDataset

# --- PYSPARK CORE & DATA STRUCTS ---
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StringType, NumericType, DoubleType,
    TimestampType, IntegerType, StructType, StructField
)

# --- PYSPARK FUNCTIONS ---
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, count, when, countDistinct, count_distinct, row_number,
    radians, sin, cos, atan2, sqrt, to_timestamp, unix_timestamp, hour,
    broadcast, monotonically_increasing_id, mean, pow
)
from pyspark.sql.functions import abs as _abs
from pyspark.sql.functions import sum as _sum

# --- PYSPARK MACHINE LEARNING (MLlib) ---
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, StandardScaler, VectorAssembler,
    VarianceThresholdSelector, UnivariateFeatureSelector
)
from pyspark.ml.stat import Correlation
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# --- PYSPARK DISTRIBUTED DEEP LEARNING ---
from pyspark.ml.torch.distributor import TorchDistributor

### Spark session builder

In [ ]:
spark = SparkSession.builder \
    .appName("RandomForest Training") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

26/05/31 16:36:06 WARN Utils: Your hostname, comrade resolves to a loopback address: 127.0.1.1; using 192.168.1.213 instead (on interface wlo1)
26/05/31 16:36:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/31 16:36:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/05/31 16:36:18 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


### First look at the data

Here we are uploading our train dataset, it's a CSV file of 50000 rows selected randomly from the dataset from the month of January.

In [5]:
file_location = "train.csv"
file_type = "csv"

df = spark.read.format(file_type) \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .option("sep", ",") \
    .load(file_location)

In [6]:
df.limit(5).show()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       1| 2016-01-20 08:54:51|  2016-01-20 09:08:34|              1|          1.3|-73.97099304199219| 40.75813674926758|         1|    

Here we can see the shape of our dataset

In [7]:
total_rows = df.count()
total_columns = len(df.columns)

print((total_rows, total_columns))

(50000, 19)


And the schema

In [8]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



We check for null values

In [9]:
optimized_null_counts = (
    df.select([
        _sum(
            when(
                col(c).isNull() | (col(c).cast("string") == "null"), 
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ])
).first().asDict()

print(optimized_null_counts)

{'VendorID': 0, 'tpep_pickup_datetime': 0, 'tpep_dropoff_datetime': 0, 'passenger_count': 0, 'trip_distance': 0, 'pickup_longitude': 0, 'pickup_latitude': 0, 'RatecodeID': 0, 'store_and_fwd_flag': 0, 'dropoff_longitude': 0, 'dropoff_latitude': 0, 'payment_type': 0, 'fare_amount': 0, 'extra': 0, 'mta_tax': 0, 'tip_amount': 0, 'tolls_amount': 0, 'improvement_surcharge': 0, 'total_amount': 0}


And duplicates

In [10]:
duplicate_count = df.select(
    count(lit(1)) - countDistinct(*df.columns)
).first()[0]

print(duplicate_count if duplicate_count is not None else 0)

26/05/30 13:00:05 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


0


Since we have no duplicates we can start looking at our unique values

In [11]:
optimized_distinct_counts = (
    df.select([
        count_distinct(col(c)).alias(c) 
        for c in df.columns
    ])
).first().asDict()

for column, distinct_count in optimized_distinct_counts.items():
    print(f"{column:<20} {distinct_count}")

VendorID             2
tpep_pickup_datetime 49420
tpep_dropoff_datetime 49375
passenger_count      7
trip_distance        1888
pickup_longitude     10876
pickup_latitude      20267
RatecodeID           7
store_and_fwd_flag   2
dropoff_longitude    12551
dropoff_latitude     22495
payment_type         4
fare_amount          216
extra                6
mta_tax              3
tip_amount           814
tolls_amount         41
improvement_surcharge 3
total_amount         1597


In this setting we really care about unique values in our categorical columns, specifically:
- Ratecode ID → from the Kaggle page we know that gives us information about the tax regime that will be applied on the fare based on geographical zones


From the table we can see that the majority of other columns are numerical, we will extrapolate categorical information from them in the following section.

## Cleaning datasets

#### Training

From the dataset we are removing the following columns since by looking at the metadata we thought that they won't be useful for our investigation. Also, some of them, if used would imply a certain level of data leakage that will make our models unreliable.

- VendorID -> not related to what we want to predict (A code indicating the TPEP provider that provided the record)
- store_and_fwd -> not related to what we want to predict (This flag indicates whether the trip record was held in vehicle memory before sending to the vendor,
aka “store and forward,” because the vehicle did not have a connection to the server)
- payment_type -> not related to what we wanted to investigate (A numeric code signifying how the passenger paid for the trip)
- passenger_count -> not related to what we wanted to investigate (The number of passengers in the vehicle. This is a driver-entered value)
- total_amount -> data leakage since it will be too directly connected to the fare and to the time (The total amount charged to passengers. Does not include cash tips.)
- tip_amount -> not related to what we want to investigate, also is only for a small subsample of the rides (This field is automatically populated for credit card tips.Cash tips are not included.)

In [12]:
columns_to_drop = [
    "VendorID",
    "store_and_fwd_flag",
    "payment_type",
    "passenger_count",
    "total_amount",
    "tip_amount"
]

df = df.drop(*columns_to_drop)

Here we are extracting the time in seconds for the trips based on the pickup and drop-off hour and the hour when the pickup was done. We will also remove the values that are less than 0 seconds and greater than 4 hours

In [13]:
date_format = "yyyy-MM-dd HH:mm:ss"

df = (
    df.withColumn("pickup_ts", to_timestamp(col("tpep_pickup_datetime"), date_format))
      .withColumn("dropoff_ts", to_timestamp(col("tpep_dropoff_datetime"), date_format))
      .withColumn("target_time", col("dropoff_ts").cast("long") - col("pickup_ts").cast("long"))
      .withColumn("pickup_hour", hour(col("pickup_ts")))
      .filter((col("target_time") > 0) & (col("target_time") < 14400))
)

df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "target_time", "pickup_hour").show(5)

+--------------------+---------------------+-----------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|target_time|pickup_hour|
+--------------------+---------------------+-----------+-----------+
| 2016-01-20 08:54:51|  2016-01-20 09:08:34|        823|          8|
| 2016-01-27 19:13:57|  2016-01-27 19:32:39|       1122|         19|
| 2016-01-16 11:34:09|  2016-01-16 11:44:50|        641|         11|
| 2016-01-22 19:11:33|  2016-01-22 19:19:47|        494|         19|
| 2016-01-03 14:44:59|  2016-01-03 14:57:20|        741|         14|
+--------------------+---------------------+-----------+-----------+
only showing top 5 rows



In [14]:
total_rows = df.count()
total_columns = len(df.columns)

print((total_rows, total_columns))

(49871, 17)


We drop the pickup datetime field

In [15]:
df = df.drop("tpep_pickup_datetime")

Now we take a look at our numerical data distribution

In [16]:
numeric_cols = ["trip_distance", "fare_amount", "extra", "mta_tax", 
             "tolls_amount", "improvement_surcharge", "target_time", "pickup_hour"]

In [17]:
optimized_summary = df.select(numeric_cols).summary("count", "mean", "stddev", "min", "max")
optimized_summary.limit(10).show()

26/05/30 13:00:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+-------------------+-------------------+---------------------+-----------------+------------------+
|summary|     trip_distance|       fare_amount|             extra|            mta_tax|       tolls_amount|improvement_surcharge|      target_time|       pickup_hour|
+-------+------------------+------------------+------------------+-------------------+-------------------+---------------------+-----------------+------------------+
|  count|             49871|             49871|             49871|              49871|              49871|                49871|            49871|             49871|
|   mean|2.8967863086763774|12.430471416253932| 0.314511439513946| 0.4978945679854023|0.30767059012251446|  0.29975336367841315|788.6021535561749|13.578713079745745|
| stddev|3.6306779724228284|10.705954205043394|0.3664381002514568|0.03491056195254574|  4.586435225187836| 0.011937749079406661|618.9985723985297| 6.382398158401388|
|   

From this table we can see many problematic values related to the absolute value or the scale of the data. We will take care of them in the next section for our train, validation and test sets. 

##### Trip distance correction

This step replaces invalid values in the `trip_distance` column (zero or null) with a computed Haversine distance based on pickup and dropoff coordinates. Otherwise, it keeps the original value unchanged.

In [18]:
EARTH_RADIUS = 6371.0

lat1 = radians(col("pickup_latitude"))
lon1 = radians(col("pickup_longitude"))
lat2 = radians(col("dropoff_latitude"))
lon2 = radians(col("dropoff_longitude"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
c_val = 2 * atan2(sqrt(a), sqrt(1 - a))
haversine_dist = EARTH_RADIUS * c_val


df = (
    df.withColumn(
        "trip_distance",
        when(
            (col("trip_distance") <= 0) | col("trip_distance").isNull(),
            haversine_dist,
        ).otherwise(col("trip_distance")),
    )
)

df.select("trip_distance", "pickup_latitude", "dropoff_latitude").limit(5).show()

+-------------+------------------+------------------+
|trip_distance|   pickup_latitude|  dropoff_latitude|
+-------------+------------------+------------------+
|          1.3| 40.75813674926758| 40.74278259277344|
|          2.0| 40.77630615234375| 40.76253128051758|
|         2.88|40.713096618652344|40.748783111572266|
|          1.1|    40.78466796875| 40.77330017089844|
|          1.5|   40.780029296875| 40.76451110839844|
+-------------+------------------+------------------+



##### Area imputing

To help our models converging and to reduce the learning time we decided to encode the coordinates using some important New York hot spots. The rows will be connected to their respective nearest hotspot

In [19]:
NYC_HOTSPOTS = {
        "JFK_Airport":      (-73.7781, 40.6413),
        "LaGuardia_Airport":(-73.8740, 40.7769),
        "Times_Square":     (-73.9851, 40.7589),
        "Empire_State":     (-73.9857, 40.7484),
        "Central_Park":     (-73.9654, 40.7829),
        "Wall_Street":      (-74.0113, 40.7074),
        "Brooklyn_Bridge":  (-73.9969, 40.7061),
        "Williamsburg":     (-73.9571, 40.7081),
        "Astoria":          (-73.9235, 40.7644),
        "Long_Island_City": (-73.9388, 40.7505),
        "Yankee_Stadium":   (-73.9262, 40.8296),
        "Harlem":           (-73.9465, 40.8116),
    }

In [20]:
hotspots_data = [(name, coords[0], coords[1]) for name, coords in NYC_HOTSPOTS.items()]
hotspots_df = spark.createDataFrame(hotspots_data, ["area_name", "h_lon", "h_lat"])

def match_areas(input_df, lat_col, lon_col, final_name):
    """
Matches each coordinate pair (latitude, longitude) in the input dataframe
with the nearest hotspot area using Euclidean distance.

The function:
1. Adds a temporary unique identifier to each row
2. Performs a cross join with the hotspot dataframe
3. Computes the distance between trip coordinates and hotspot coordinates
4. Selects the nearest hotspot using a window function
5. Returns the dataframe with the matched area name
"""
    temp_id_df = input_df.withColumn("_internal_id", monotonically_increasing_id())
    
    joined_df = temp_id_df.crossJoin(broadcast(hotspots_df)).withColumn(
        "dist", 
        sqrt(pow(col(lon_col) - col("h_lon"), 2) + pow(col(lat_col) - col("h_lat"), 2))
    )
    
    window_spec = Window.partitionBy("_internal_id").orderBy("dist")
    
    return (
        joined_df.withColumn("rn", row_number().over(window_spec))
        .filter(col("rn") == 1)
        .drop("h_lon", "h_lat", "dist", "rn", "_internal_id")
        .withColumnRenamed("area_name", final_name)
    )


df = match_areas(df, "pickup_latitude", "pickup_longitude", "area_pickup")
df = match_areas(df, "dropoff_latitude", "dropoff_longitude", "area_dropoff")

cols_to_remove = ["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"]
df = df.drop(*cols_to_remove)

df.limit(5).show()

+---------------------+-------------+----------+-----------+-----+-------+------------+---------------------+-------------------+-------------------+-----------+-----------+------------+---------------+
|tpep_dropoff_datetime|trip_distance|RatecodeID|fare_amount|extra|mta_tax|tolls_amount|improvement_surcharge|          pickup_ts|         dropoff_ts|target_time|pickup_hour| area_pickup|   area_dropoff|
+---------------------+-------------+----------+-----------+-----+-------+------------+---------------------+-------------------+-------------------+-----------+-----------+------------+---------------+
|  2016-01-20 09:08:34|          1.3|         1|        9.5|  0.0|    0.5|         0.0|                  0.3|2016-01-20 08:54:51|2016-01-20 09:08:34|        823|          8|Times_Square|   Empire_State|
|  2016-01-04 08:37:32|         2.03|         1|        8.5|  0.0|    0.5|         0.0|                  0.3|2016-01-04 08:28:59|2016-01-04 08:37:32|        513|          8|Empire_State|Br

In [21]:
print("Current columns:", df.columns)

Current columns: ['tpep_dropoff_datetime', 'trip_distance', 'RatecodeID', 'fare_amount', 'extra', 'mta_tax', 'tolls_amount', 'improvement_surcharge', 'pickup_ts', 'dropoff_ts', 'target_time', 'pickup_hour', 'area_pickup', 'area_dropoff']


##### Problematic numerical values

Here we will correct the problematic values in the following way

- Fare amount values will be transformed by getting their absolute value to ensure no negatives
- Extra values (Miscellaneous extras and surcharges. Currently, this only includes the $0.50 and $1 rush hour and overnight charges.) will be transformed using their absolute value

For mta tax (0.50 MTA tax that is automatically triggered based on the metered rate in use) and improvement surcharge (0.30 improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015) we will set their values as they are. We will not use these 2 values, but we will use them to show that our filter functions are effective in a later section. 

In [22]:
# Apply transformations to the financial columns
df = (
    df.withColumn("fare_amount", _abs(col("fare_amount")))
      .withColumn("extra", _abs(col("extra")))
      .withColumn("mta_tax", lit(0.5))
      .withColumn("improvement_surcharge", lit(0.3))
)

df.select("fare_amount", "extra", "mta_tax", "improvement_surcharge").limit(5).show()

+-----------+-----+-------+---------------------+
|fare_amount|extra|mta_tax|improvement_surcharge|
+-----------+-----+-------+---------------------+
|        9.5|  0.0|    0.5|                  0.3|
|       13.0|  1.0|    0.5|                  0.3|
|       11.0|  0.0|    0.5|                  0.3|
|        7.0|  1.0|    0.5|                  0.3|
|        9.5|  0.0|    0.5|                  0.3|
+-----------+-----+-------+---------------------+



We will now remove rows where our target values are equal to 0

In [23]:
df = df.filter((col("target_time") > 0) & (col("fare_amount") > 0))

print(f"Remaining rows: {df.count()}")

Remaining rows: 49863


And where the values for distance and tolls are unrealistically high

In [24]:
df = df.filter(
    (col("trip_distance") < 100) & 
    (col("tolls_amount") < 50)
)

This cell identifies numeric columns and applies distributed outlier detection using the IQR method on non-airport trips only. It computes approximate quantiles and global means in parallel, then replaces extreme values above the learned upper bounds with the corresponding column mean, while preserving airport-related trips unchanged.

In [25]:
excluded_cols = ["fare_amount", "target_time", "RatecodeID", "pickup_hour"]

numeric_cols = [
    f.name for f in df.schema.fields 
    if isinstance(f.dataType, NumericType) and f.name not in excluded_cols
]

IQR_MULTIPLIER = 3.0 

df = df.withColumn(
    "is_airport_trip",
    col("RatecodeID").isin(2, 3) | 
    col("area_pickup").isin("JFK_Airport", "LaGuardia_Airport") | 
    col("area_dropoff").isin("JFK_Airport", "LaGuardia_Airport")
)

df_urban_only = df.filter(~col("is_airport_trip")).cache()

all_quantiles = df_urban_only.approxQuantile(numeric_cols, [0.25, 0.75], 0.05)
mean_exprs = [mean(col_name).alias(f"{col_name}_mean") for col_name in numeric_cols]
global_means = df_urban_only.select(mean_exprs).first().asDict()

df_urban_only.unpersist()


learned_parameters = {}
for i, col_name in enumerate(numeric_cols):
    quantiles = all_quantiles[i]
    if len(quantiles) < 2: 
        continue
    
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1
    
    upper_bound = q3 + IQR_MULTIPLIER * iqr if iqr > 0 else q3 * 2.0
    if upper_bound == 0:
        upper_bound = 0.01 
        
    mean_val = global_means[f"{col_name}_mean"]
    learned_parameters[col_name] = {"upper_bound": upper_bound, "mean": mean_val}


updated_columns = {
    col_name: when(col("is_airport_trip"), col(col_name))
              .when(col(col_name) > params["upper_bound"], params["mean"])
              .otherwise(col(col_name))
    for col_name, params in learned_parameters.items()
}


df = df.withColumns(updated_columns)

Let's see our new value distribution

In [26]:
optimized_summary = df.select(numeric_cols).summary("count", "mean", "stddev", "min", "max")
optimized_summary.limit(10).show()

+-------+------------------+------------------+-------+------------------+---------------------+
|summary|     trip_distance|             extra|mta_tax|      tolls_amount|improvement_surcharge|
+-------+------------------+------------------+-------+------------------+---------------------+
|  count|             49841|             49841|  49841|             49841|                49841|
|   mean|2.7117254687966392|0.3148411950001003|    0.5|0.2559223486007885|  0.29999999999974136|
| stddev| 3.445815026729438|0.3663053563986901|    0.0| 1.248905511885673|                  0.0|
|    min|               0.0|               0.0|    0.5|               0.0|                  0.3|
|    max|             70.08|               1.5|    0.5|             30.04|                  0.3|
+-------+------------------+------------------+-------+------------------+---------------------+



#### Validation

We will apply the same preprocessing pipeline to our validation set. The validation set was taken by randomly selecting 10000 rows from the January month. We took care of avoiding duplicates between the train and validation by building first a dataframe of 60000 random rows and by splitting them later

In [27]:
file_location = "validation.csv"

df_val = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .load(file_location)
)

df_val.limit(5).show()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2016-01-29 18:03:25|  2016-01-29 18:29:27|              1|         4.91| -73.9696273803711|  40.7901611328125|         1|    

In [28]:
validation_row_count = df_val.count()
print(validation_row_count)

10000


In [29]:
columns_to_drop = ["VendorID", "store_and_fwd_flag", "payment_type", "tip_amount", "total_amount", "passenger_count"]
df_val = df_val.drop(*columns_to_drop)

In [30]:
date_format = "yyyy-MM-dd HH:mm:ss"

df_val = (
    df_val.withColumn("pickup_ts", to_timestamp(col("tpep_pickup_datetime"), date_format))
    .withColumn("dropoff_ts", to_timestamp(col("tpep_dropoff_datetime"), date_format))
    .withColumn("target_time", unix_timestamp(col("dropoff_ts")) - unix_timestamp(col("pickup_ts")))
    .withColumn("pickup_hour", hour(col("pickup_ts")))
    .filter((col("target_time") > 30) & (col("target_time") < 14400))
    .drop("tpep_dropoff_datetime", "dropoff_ts", "pickup_ts")
)

print("Check distribuzione Target Time nel Validation:")
df_val.select("target_time").summary("count", "mean", "stddev", "min", "max").limit(10).show()

Check distribuzione Target Time nel Validation:
+-------+-----------------+
|summary|      target_time|
+-------+-----------------+
|  count|             9932|
|   mean|804.9795610149014|
| stddev|626.7363771487455|
|    min|               31|
|    max|             6926|
+-------+-----------------+



In [31]:
columns_to_drop = ["tpep_pickup_datetime"]
df_val = df_val.drop(*columns_to_drop)

In [32]:
optimized_val_summary = df_val.select(numeric_cols).summary("count", "mean", "stddev", "min", "max")
optimized_val_summary.limit(10).show()

+-------+------------------+-------------------+-------------------+------------------+---------------------+
|summary|     trip_distance|              extra|            mta_tax|      tolls_amount|improvement_surcharge|
+-------+------------------+-------------------+-------------------+------------------+---------------------+
|  count|              9932|               9932|               9932|              9932|                 9932|
|   mean| 2.942862464760359|0.31463954893274265| 0.4979359645590012|0.3006363270237602|   0.2997281514297568|
| stddev|3.9609322010013144| 0.3677065089003221|0.03506066964824858|1.3656781273046628| 0.012409235506445203|
|    min|               0.0|               -1.0|               -0.5|               0.0|                 -0.3|
|    max|             176.5|                1.0|                0.5|              21.0|                  0.3|
+-------+------------------+-------------------+-------------------+------------------+---------------------+



In [33]:
df_val = df_val.withColumn(
    "trip_distance",
    when(
        (col("trip_distance") <= 0) | col("trip_distance").isNull(),
        haversine_dist
    ).otherwise(col("trip_distance"))
)

df_val.select(
    "trip_distance", 
    "pickup_latitude", 
    "pickup_longitude", 
    "dropoff_latitude", 
    "dropoff_longitude"
).limit(5).show()

+-------------+------------------+------------------+------------------+------------------+
|trip_distance|   pickup_latitude|  pickup_longitude|  dropoff_latitude| dropoff_longitude|
+-------------+------------------+------------------+------------------+------------------+
|         4.91|  40.7901611328125| -73.9696273803711|40.747596740722656|-74.00589752197266|
|          0.9| 40.76517868041992|-73.96623992919922| 40.75639343261719|-73.96371459960938|
|          0.6| 40.74167251586914|-74.00475311279297| 40.73759841918945|-73.99745178222656|
|          6.4|40.764862060546875|-73.98799133300781| 40.81203842163086|-73.93557739257812|
|          1.1| 40.74464797973633|-73.97593688964844|40.746177673339844|-73.99272155761719|
+-------------+------------------+------------------+------------------+------------------+



In [34]:
df_val = match_areas(df_val, "pickup_latitude", "pickup_longitude", "area_pickup")
df_val = match_areas(df_val, "dropoff_latitude", "dropoff_longitude", "area_dropoff")

cols_to_remove = ["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"]
df_val = df_val.drop(*cols_to_remove)

df_val.limit(5).show()

+-------------+----------+-----------+-----+-------+------------+---------------------+-----------+-----------+------------+------------+
|trip_distance|RatecodeID|fare_amount|extra|mta_tax|tolls_amount|improvement_surcharge|target_time|pickup_hour| area_pickup|area_dropoff|
+-------------+----------+-----------+-----+-------+------------+---------------------+-----------+-----------+------------+------------+
|         4.91|         1|       20.0|  1.0|    0.5|         0.0|                  0.3|       1562|         18|Central_Park|Empire_State|
|          0.9|         1|        5.0|  0.0|    0.5|         0.0|                  0.3|        205|          7|Central_Park|Times_Square|
|          0.6|         1|        8.0|  0.0|    0.5|         0.0|                  0.3|        707|         12|Empire_State|Empire_State|
|          6.4|         1|       20.5|  0.5|    0.5|         0.0|                  0.3|       1078|          1|Times_Square|      Harlem|
|          1.1|         1|       1

In [35]:
df_val = (
    df_val.withColumn("fare_amount", _abs(col("fare_amount")))
    .withColumn("extra", _abs(col("extra")))
    .withColumn("mta_tax", lit(0.5))
    .withColumn("improvement_surcharge", lit(0.3))
)

df_val.select("fare_amount", "extra", "mta_tax", "improvement_surcharge").limit(5).show()

+-----------+-----+-------+---------------------+
|fare_amount|extra|mta_tax|improvement_surcharge|
+-----------+-----+-------+---------------------+
|       20.0|  1.0|    0.5|                  0.3|
|        5.0|  0.0|    0.5|                  0.3|
|        8.0|  0.0|    0.5|                  0.3|
|       20.5|  0.5|    0.5|                  0.3|
|       10.5|  0.0|    0.5|                  0.3|
+-----------+-----+-------+---------------------+



In [36]:
df_val = df_val.filter(
    (col("target_time") > 0) & 
    (col("fare_amount") > 0) & 
    (col("trip_distance") > 0)
)

print(f"Remaining rows: {df_val.count()}")

Remaining rows: 9925


In [37]:
optimized_val_summary = df_val.select(numeric_cols).summary("count", "mean", "stddev", "min", "max")
optimized_val_summary.limit(10).show()

+-------+--------------------+------------------+-------+-------------------+---------------------+
|summary|       trip_distance|             extra|mta_tax|       tolls_amount|improvement_surcharge|
+-------+--------------------+------------------+-------+-------------------+---------------------+
|  count|                9925|              9925|   9925|               9925|                 9925|
|   mean|  2.9451287171204155|0.3152141057934509|    0.5|0.30084836272040166|  0.30000000000003474|
| stddev|  3.9614384608882407|0.3674046295740516|    0.0| 1.3661363410348317|                  0.0|
|    min|7.701503056143404E-4|               0.0|    0.5|                0.0|                  0.3|
|    max|               176.5|               1.0|    0.5|               21.0|                  0.3|
+-------+--------------------+------------------+-------+-------------------+---------------------+



In [38]:
df_val = df_val.filter(
    (F.col("trip_distance") < 100) &  
    (F.col("tolls_amount") < 50)      
)

Here we will apply the values learned on our training set on the validation one

In [39]:
validation_updates = {
    col_name: when(col(col_name) > params["upper_bound"], params["mean"]).otherwise(col(col_name))
    for col_name, params in learned_parameters.items()
    if col_name in df_val.columns
}

for col_name in validation_updates.keys():
    print(f"Validation Adjusted {col_name} using Train values.")

df_val = df_val.withColumns(validation_updates)

print("\nValidation Sample:")
df_val.show(5)

Validation Adjusted trip_distance using Train values.
Validation Adjusted extra using Train values.
Validation Adjusted mta_tax using Train values.
Validation Adjusted tolls_amount using Train values.
Validation Adjusted improvement_surcharge using Train values.

Validation Sample:
+-------------+----------+-----------+-----+-------+------------+---------------------+-----------+-----------+------------+------------+
|trip_distance|RatecodeID|fare_amount|extra|mta_tax|tolls_amount|improvement_surcharge|target_time|pickup_hour| area_pickup|area_dropoff|
+-------------+----------+-----------+-----+-------+------------+---------------------+-----------+-----------+------------+------------+
|         4.91|         1|       20.0|  1.0|    0.5|         0.0|                  0.3|       1562|         18|Central_Park|Empire_State|
|          0.9|         1|        5.0|  0.0|    0.5|         0.0|                  0.3|        205|          7|Central_Park|Times_Square|
|          0.6|         1| 

In [40]:
optimized_val_summary = df_val.select(numeric_cols).summary("count", "mean", "stddev", "min", "max")
optimized_val_summary.limit(10).show()

+-------+--------------------+-------------------+-------+--------------------+---------------------+
|summary|       trip_distance|              extra|mta_tax|        tolls_amount|improvement_surcharge|
+-------+--------------------+-------------------+-------+--------------------+---------------------+
|  count|                9924|               9924|   9924|                9924|                 9924|
|   mean|  1.9617337037900386|0.31524586860137044|    0.5| 0.00177622083643179|  0.30000000000003474|
| stddev|  1.3118270842347541|0.36740951415479994|    0.0|0.007671416315874478|                  0.0|
|    min|7.701503056143404E-4|                0.0|    0.5|                 0.0|                  0.3|
|    max|                 6.9|                1.0|    0.5|0.034905377387621804|                  0.3|
+-------+--------------------+-------------------+-------+--------------------+---------------------+



In [41]:
df = df.drop("tpep_dropoff_datetime", "pickup_ts", "dropoff_ts", "is_airport_trip")

In [42]:
df_val = df_val.drop("tpep_dropoff_datetime", "pickup_ts", "dropoff_ts", "is_airport_trip")

In [43]:
target_cols = ["target_time", "fare_amount"]

We can see the distributions of our training and validation set

In [44]:
optimized_target_summary = df.select(target_cols).summary("count", "mean", "stddev", "min", "max")
optimized_target_summary.limit(10).show()

+-------+-----------------+------------------+
|summary|      target_time|       fare_amount|
+-------+-----------------+------------------+
|  count|            49841|             49841|
|   mean| 788.858931401858|12.427863004353847|
| stddev|618.5033305379781|10.660277118167581|
|    min|                1|              0.01|
|    max|            10246|             348.0|
+-------+-----------------+------------------+



In [45]:
optimized_target_summary = df_val.select(target_cols).summary("count", "mean", "stddev", "min", "max")
optimized_target_summary.limit(10).show()

+-------+-----------------+------------------+
|summary|      target_time|       fare_amount|
+-------+-----------------+------------------+
|  count|             9924|              9924|
|   mean|805.2315598548972|12.529768238613462|
| stddev|626.1048788713542|10.300911033555655|
|    min|               31|              0.01|
|    max|             6926|             143.0|
+-------+-----------------+------------------+



#### Test

As above but for the test set. The test set was taken from the month of March

In [46]:
file_location = "test.csv"
file_type = "csv"

df_test = (
    spark.read.format(file_type)
    .option("header", "true")
    .option("sep", ",")
    .option("inferSchema", "true")  
    .load(file_location)
)

df_test.limit(5).show()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       1| 2016-03-17 19:43:51|  2016-03-17 19:53:57|            1.0|          1.8|-73.99541473388672| 40.73929977416992|       1.0|    

In [47]:
print(f"Test set row count: {df_test.count()}")

Test set row count: 20000


In [48]:
df_test = df_test.drop("VendorID", "store_and_fwd_flag", "payment_type", "tip_amount", "total_amount", "passenger_count")

In [49]:
date_format = "yyyy-MM-dd HH:mm:ss"

df_test = (
    df_test.withColumn("pickup_ts", to_timestamp(col("tpep_pickup_datetime"), date_format))
    .withColumn("dropoff_ts", to_timestamp(col("tpep_dropoff_datetime"), date_format))
    .withColumn("target_time", unix_timestamp(col("dropoff_ts")) - unix_timestamp(col("pickup_ts")))
    .withColumn("pickup_hour", hour(col("pickup_ts")))
    .filter((col("target_time") > 30) & (col("target_time") < 14400))
    .drop("tpep_dropoff_datetime", "dropoff_ts", "pickup_ts")
)


df_test.select("target_time").summary("count", "mean", "stddev", "min", "max").limit(10).show()

+-------+-----------------+
|summary|      target_time|
+-------+-----------------+
|  count|            19848|
|   mean|838.8704151551793|
| stddev|655.6146958007539|
|    min|               31|
|    max|            11132|
+-------+-----------------+



In [50]:
df_test = df_test.drop("tpep_pickup_datetime")

In [51]:
optimized_test_summary = df_test.select(numeric_cols).summary("count", "mean", "stddev", "min", "max")
optimized_test_summary.limit(10).show()

+-------+------------------+-------------------+--------------------+-------------------+---------------------+
|summary|     trip_distance|              extra|             mta_tax|       tolls_amount|improvement_surcharge|
+-------+------------------+-------------------+--------------------+-------------------+---------------------+
|  count|             19848|              19848|               19848|              19848|                19848|
|   mean|2.9621120515921033|0.35331015719467956|  0.4979468964127368|0.39024738008867227|   0.2996977025394068|
| stddev| 3.613040783114459| 0.4760427966338212|0.034939673358015774| 10.110091830141474| 0.013123522626746173|
|    min|               0.0|               -1.0|                -0.5|                0.0|                 -0.3|
|    max|              40.8|                4.5|                 0.5|            1410.32|                  0.3|
+-------+------------------+-------------------+--------------------+-------------------+---------------

In [52]:
df_test = df_test.withColumn(
    "trip_distance",
    when(
        (col("trip_distance") <= 0) | col("trip_distance").isNull(), 
        haversine_dist
    ).otherwise(col("trip_distance"))
)

df_test.select("trip_distance", "pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude").limit(5).show()

+-------------+------------------+------------------+------------------+------------------+
|trip_distance|   pickup_latitude|  pickup_longitude|  dropoff_latitude| dropoff_longitude|
+-------------+------------------+------------------+------------------+------------------+
|          1.8| 40.73929977416992|-73.99541473388672| 40.75969696044922|-73.99535369873048|
|         1.19| 40.76432037353516|-73.96170806884764|40.766498565673835|-73.97949981689453|
|          9.2|40.767234802246094|-73.95633697509764| 40.77033996582031|-73.86480712890625|
|         0.87|  40.7339096069336|-73.98069763183594|40.734317779541016|-73.99137878417967|
|          2.6| 40.73977279663086|-74.00618743896483| 40.70954513549805|-74.01781463623048|
+-------------+------------------+------------------+------------------+------------------+



In [53]:
df_test = match_areas(df_test, "pickup_latitude", "pickup_longitude", "area_pickup")
df_test = match_areas(df_test, "dropoff_latitude", "dropoff_longitude", "area_dropoff")

cols_to_remove = ["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"]
df_test = df_test.drop(*cols_to_remove)

df_test.limit(5).show()

+-------------+----------+-----------+-----+-------+------------+---------------------+-----------+-----------+------------+-----------------+
|trip_distance|RatecodeID|fare_amount|extra|mta_tax|tolls_amount|improvement_surcharge|target_time|pickup_hour| area_pickup|     area_dropoff|
+-------------+----------+-----------+-----+-------+------------+---------------------+-----------+-----------+------------+-----------------+
|          1.8|       1.0|        9.0|  1.0|    0.5|         0.0|                  0.3|        606|         19|Empire_State|     Times_Square|
|         1.19|       1.0|        6.5|  0.5|    0.5|         0.0|                  0.3|        427|         23|Central_Park|     Times_Square|
|          9.2|       1.0|       25.5|  0.0|    0.5|        5.54|                  0.3|        903|          5|Central_Park|LaGuardia_Airport|
|         0.87|       1.0|        5.5|  0.0|    0.5|         0.0|                  0.3|        300|         10|Empire_State|     Empire_State|

In [54]:
df_test = (
    df_test.withColumn("fare_amount", _abs(col("fare_amount")))
    .withColumn("extra", _abs(col("extra")))
    .withColumn("mta_tax", lit(0.5))
    .withColumn("improvement_surcharge", lit(0.3))
)

df_test.select("fare_amount", "extra", "mta_tax", "improvement_surcharge").limit(5).show()

+-----------+-----+-------+---------------------+
|fare_amount|extra|mta_tax|improvement_surcharge|
+-----------+-----+-------+---------------------+
|        9.0|  1.0|    0.5|                  0.3|
|        6.5|  0.5|    0.5|                  0.3|
|       25.5|  0.0|    0.5|                  0.3|
|        5.5|  0.0|    0.5|                  0.3|
|       12.5|  1.0|    0.5|                  0.3|
+-----------+-----+-------+---------------------+



In [55]:
df_test = df_test.filter(
    (col("target_time") > 0) & 
    (col("fare_amount") > 0) & 
    (col("trip_distance") > 0)
)

print(f"Remaining rows: {df_test.count()}")

Remaining rows: 19839


In [56]:
df_test = df_test.filter(
    (col("trip_distance") < 100) & 
    (col("tolls_amount") < 50)
)

In [57]:
test_updates = {
    col_name: when(col(col_name) > params["upper_bound"], params["mean"]).otherwise(col(col_name))
    for col_name, params in learned_parameters.items()
    if col_name in df_test.columns
}

for col_name in test_updates.keys():
    print(f"Test Adjusted {col_name} using Train values.")

df_test = df_test.withColumns(test_updates)

print("\nTest Sample:")
df_test.show(5)

Test Adjusted trip_distance using Train values.
Test Adjusted extra using Train values.
Test Adjusted mta_tax using Train values.
Test Adjusted tolls_amount using Train values.
Test Adjusted improvement_surcharge using Train values.

Test Sample:
+------------------+----------+-----------+-----+-------+--------------------+---------------------+-----------+-----------+------------+-----------------+
|     trip_distance|RatecodeID|fare_amount|extra|mta_tax|        tolls_amount|improvement_surcharge|target_time|pickup_hour| area_pickup|     area_dropoff|
+------------------+----------+-----------+-----+-------+--------------------+---------------------+-----------+-----------+------------+-----------------+
|               1.8|       1.0|        9.0|  1.0|    0.5|                 0.0|                  0.3|        606|         19|Empire_State|     Times_Square|
|              1.19|       1.0|        6.5|  0.5|    0.5|                 0.0|                  0.3|        427|         23|Centr

In [58]:
optimized_test_summary = df_test.select(numeric_cols).summary("count", "mean", "stddev", "min", "max")
optimized_test_summary.limit(10).show()

+-------+--------------------+-------------------+-------+--------------------+---------------------+
|summary|       trip_distance|              extra|mta_tax|        tolls_amount|improvement_surcharge|
+-------+--------------------+-------------------+-------+--------------------+---------------------+
|  count|               19838|              19838|  19838|               19838|                19838|
|   mean|  1.9656744348027928|0.33352785712847044|    0.5| 0.00186333272776956|   0.3000000000001083|
| stddev|  1.3033942829664416|0.37800609813773184|    0.0|0.007846746263167474|                  0.0|
|    min|4.241749828731711E-4|                0.0|    0.5|                 0.0|                  0.3|
|    max|                 6.9|                1.0|    0.5|0.034905377387621804|                  0.3|
+-------+--------------------+-------------------+-------+--------------------+---------------------+



In [59]:
df_test.printSchema()

root
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = false)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = false)
 |-- target_time: long (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- area_pickup: string (nullable = true)
 |-- area_dropoff: string (nullable = true)



In [60]:
df_test = df_test.drop("tpep_dropoff_datetime", "pickup_ts", "dropoff_ts", "is_airport_trip", "improvement_surcharge", "mta_tax", "tip_amount")

### Filter methods

Before passing the features into any model, we apply three complementary **filter methods** to assess the relevance and statistical significance of each candidate feature. Filter methods are model-agnostic and computationally inexpensive — they evaluate each feature independently of the learning algorithm.

We evaluate five numerical features against both prediction targets:

The three filter methods applied are:
1. **Variance Threshold** — drops features whose variance is too low to carry meaningful signal
2. **Univariate Feature Selection (F-test / FPR)** — retains only features whose F-statistic is statistically significant against each target
3. **Spearman Rank Correlation** — retains features with a monotonic correlation above a minimum threshold


In [61]:
TARGETS = ["target_time", "fare_amount"]

feature_cols = [
    "trip_distance", 
    "extra", "tolls_amount", "improvement_surcharge", "mta_tax"
]

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Targets: {TARGETS}")

Features (5): ['trip_distance', 'extra', 'tolls_amount', 'improvement_surcharge', 'mta_tax']
Targets: ['target_time', 'fare_amount']


In [62]:
def assemble(dataframe, features, output_col="features"):
    return VectorAssembler(
        inputCols=features,
        outputCol=output_col,
        handleInvalid="skip"
    ).transform(dataframe)

#### 1. Variance Threshold

A feature with very low variance is nearly constant across all samples and carries almost no predictive information. We set a threshold of **0.01**: any feature whose variance falls below this value is removed before model training.

`VarianceThresholdSelector` from PySpark MLlib computes per-feature variance on the assembled feature vector and drops low-variance features automatically.


In [63]:
VARIANCE_THRESHOLD = 0.01

df_vec = assemble(df, feature_cols)

vt_selector = VarianceThresholdSelector(
    featuresCol="features",
    outputCol="selected_features",
    varianceThreshold=VARIANCE_THRESHOLD
)
vt_model = vt_selector.fit(df_vec)

selected_indices = vt_model.selectedFeatures
dropped_indices  = [i for i in range(len(feature_cols)) if i not in selected_indices]

passed_variance  = [feature_cols[i] for i in selected_indices]
dropped_variance = [feature_cols[i] for i in dropped_indices]

print(f"Passed  ({len(passed_variance)}): {passed_variance}")
print(f"Dropped ({len(dropped_variance)}): {dropped_variance}")

Passed  (3): ['trip_distance', 'extra', 'tolls_amount']
Dropped (2): ['improvement_surcharge', 'mta_tax']


#### 2. Univariate Feature Selection (F-test)

For each feature–target pair we run an **F-test** and keep only those features whose p-value falls below the false positive rate (FPR) threshold of **0.05**. This ensures that the association between each feature and the target is statistically significant before it reaches the models.

`UnivariateFeatureSelector` is configured with `featureType="continuous"` and `labelType="continuous"`, which triggers an ANOVA F-test under the hood. The selection is run independently for each target (`target_time` and `fare_amount`).


In [64]:
FPR_THRESHOLD = 0.05 #pvalue

univariate_results = {}

for target in TARGETS:
    df_target = assemble(df.select(feature_cols + [target]), feature_cols)

    selector = UnivariateFeatureSelector(
        featuresCol="features",
        outputCol="selected_features",
        labelCol=target
    )
    selector.setFeatureType("continuous")
    selector.setLabelType("continuous")
    selector.setSelectionMode("fpr")
    selector.setSelectionThreshold(FPR_THRESHOLD)

    model = selector.fit(df_target)

    selected_idx = model.selectedFeatures
    dropped_idx  = [i for i in range(len(feature_cols)) if i not in selected_idx]
    passed       = [feature_cols[i] for i in selected_idx]
    dropped      = [feature_cols[i] for i in dropped_idx]

    univariate_results[target] = {"passed": passed, "dropped": dropped}

    print(f"\n── {target} ──")
    print(f"  Passed  ({len(passed)}): {passed}")
    print(f"  Dropped ({len(dropped)}): {dropped}")


── target_time ──
  Passed  (3): ['trip_distance', 'tolls_amount', 'extra']
  Dropped (2): ['improvement_surcharge', 'mta_tax']



── fare_amount ──
  Passed  (3): ['trip_distance', 'extra', 'tolls_amount']
  Dropped (2): ['improvement_surcharge', 'mta_tax']


#### 3. Spearman Rank Correlation

Unlike Pearson correlation, **Spearman's ρ** measures *monotonic* relationships and is robust to outliers and non-normal distributions, both common in taxi-trip data. We retain features whose absolute Spearman correlation with the target is at least **0.05**.

The full correlation matrix is computed via `Correlation.corr(..., method="spearman")` in PySpark, and the row corresponding to each target is extracted to rank features.


In [65]:
SPEARMAN_THRESHOLD = 0.05

spearman_results = {}

for target in TARGETS:
    df_target = assemble(df.select(feature_cols + [target]), feature_cols + [target])

    corr_matrix  = Correlation.corr(df_target, "features", method="spearman").head()[0]
    corr_array   = corr_matrix.toArray()
    target_corrs = corr_array[len(feature_cols), :len(feature_cols)]

    sp_df = pd.DataFrame({
        "feature":    feature_cols,
        "spearman_r": target_corrs,
        "abs_r":      np.abs(target_corrs)
    }).sort_values("abs_r", ascending=False)

    passed  = sp_df[sp_df["abs_r"] >= SPEARMAN_THRESHOLD]["feature"].tolist()
    dropped = sp_df[sp_df["abs_r"] <  SPEARMAN_THRESHOLD]["feature"].tolist()
    spearman_results[target] = {"df": sp_df, "passed": passed, "dropped": dropped}

    print(f"\n── {target} ──")
    print(sp_df.to_string(index=False))
    print(f"  Passed  ({len(passed)}): {passed}")
    print(f"  Dropped ({len(dropped)}): {dropped}")

26/05/30 13:01:28 WARN Executor: Managed memory leak detected; size = 5248132 bytes, task 0.0 in stage 292.0 (TID 972)
26/05/30 13:01:28 WARN Executor: Managed memory leak detected; size = 5248132 bytes, task 0.0 in stage 302.0 (TID 975)
26/05/30 13:01:31 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/05/30 13:01:31 WARN PearsonCorrelation: Pearson correlation matrix contains NaN values.



── target_time ──
              feature  spearman_r    abs_r
        trip_distance    0.834674 0.834674
         tolls_amount    0.317974 0.317974
                extra   -0.023049 0.023049
improvement_surcharge         NaN      NaN
              mta_tax         NaN      NaN
  Passed  (2): ['trip_distance', 'tolls_amount']
  Dropped (1): ['extra']


26/05/30 13:01:35 WARN Executor: Managed memory leak detected; size = 16230890 bytes, task 0.0 in stage 316.0 (TID 992)
26/05/30 13:01:36 WARN Executor: Managed memory leak detected; size = 16230890 bytes, task 0.0 in stage 326.0 (TID 994)



── fare_amount ──
              feature  spearman_r    abs_r
        trip_distance    0.914288 0.914288
         tolls_amount    0.354552 0.354552
                extra   -0.011808 0.011808
improvement_surcharge         NaN      NaN
              mta_tax         NaN      NaN
  Passed  (2): ['trip_distance', 'tolls_amount']
  Dropped (1): ['extra']


26/05/30 13:01:37 WARN PearsonCorrelation: Pearson correlation matrix contains NaN values.


#### Results Summary

The table below consolidates the outcome of all three filter methods for each feature–target pair. A **✓** means the feature passed the filter; a **✗** means it was dropped.


In [66]:
rows = []
for feat in feature_cols:
    for target in TARGETS:
        v1 = "✓" if feat in passed_variance                      else "✗"
        v2 = "✓" if feat in univariate_results[target]["passed"] else "✗"
        v3 = "✓" if feat in spearman_results[target]["passed"]   else "✗"
        rows.append({
            "target":     target,
            "feature":    feat,
            "variance":   v1,
            "univariate": v2,
            "spearman":   v3,
        })

summary_df = pd.DataFrame(rows)

for target in TARGETS:
    print(f"\n{'='*55}")
    print(f"  {target}")
    print(f"{'='*55}")
    t = summary_df[summary_df["target"] == target][["feature","variance","univariate","spearman"]]
    print(t.to_string(index=False))


  target_time
              feature variance univariate spearman
        trip_distance        ✓          ✓        ✓
                extra        ✓          ✓        ✗
         tolls_amount        ✓          ✓        ✓
improvement_surcharge        ✗          ✗        ✗
              mta_tax        ✗          ✗        ✗

  fare_amount
              feature variance univariate spearman
        trip_distance        ✓          ✓        ✓
                extra        ✓          ✓        ✗
         tolls_amount        ✓          ✓        ✓
improvement_surcharge        ✗          ✗        ✗
              mta_tax        ✗          ✗        ✗


Based on these results, `trip_distance`, `tolls_amount`, and `extra` were selected as the numerical features for all subsequent models, while `improvement_surcharge` and `mta_tax` were discarded due to their constant values yielding zero variance.

## Models for time

In this section, we will evaluate the performance of three different Machine Learning models to predict our target variable: **time**. 

To prepare our dataset for the models, we implemented a robust PySpark preprocessing pipeline consisting of the following stages:
* **`StringIndexer` & `OneHotEncoder`:** Categorical columns are first converted into numerical indexes and then expanded into sparse binary vectors, preventing the model from assuming an artificial ordering between categories.
* **`VectorAssembler` (Numeric) & `StandardScaler`:** Numeric columns are grouped together and normalized. By setting `withMean=False`, we preserve the sparsity of the matrix, optimizing memory usage and computational efficiency.
* **Final `VectorAssembler`:** All processed features (scaled numeric and one-hot encoded categorical) are merged into a single unified column named `features`, which serves as the standard input format for Spark MLlib and deep learning frameworks.

To establish a solid benchmark, we will test the **baseline version** (using default hyperparameters) of three main architectures:

1. **Linear Regression:** Serves as our simplest baseline model to capture linear relationships and provide a performance floor.
2. **Random Forest:** A powerful tree-based ensemble method, widely recognized as one of the best choices for structured and tabular data.
3. **PyTorch-based Neural Network:** A deep learning approach integrated into the workflow to demonstrate how to leverage PyTorch's flexibility directly within a distributed Spark environment.

Each model's predictions will be evaluated and compared using standard regression metrics to identify the most effective approach for our dataset.

In [67]:
categorical_cols = ["RatecodeID", "area_pickup", "area_dropoff", "pickup_hour"]
numeric_cols = ["trip_distance", "tolls_amount", "extra"]


target_col = "target_time"

### Linear regression Baseline

As we can see from the notebook on the data distribution our values are really skewed, for this reason we might expect really poor results from a linear regression fitted on them.

In [68]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep")
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_index" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    dropLast=True
)

numeric_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="numeric_vec"
)

scaler = StandardScaler(
    inputCol="numeric_vec",
    outputCol="numeric_scaled",
    withMean=False,  
    withStd=True
)

final_assembler = VectorAssembler(
    inputCols=["numeric_scaled"] + [f"{c}_ohe" for c in categorical_cols],
    outputCol="features"
)

lr = LinearRegression(
    featuresCol="features",
    labelCol=target_col
)

pipeline = Pipeline(stages=[
    *indexers,
    encoder,
    numeric_assembler,
    scaler,
    final_assembler,
    lr
])

model = pipeline.fit(df)

train_pred = model.transform(df)
val_pred   = model.transform(df_val)

evaluator = RegressionEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="rmse"
)

rmse_val = evaluator.evaluate(val_pred)
print(f"\n[Validation] RMSE: {rmse_val:.2f}")

r2_val = evaluator.setMetricName("r2").evaluate(val_pred)
print(f"[Validation] R2: {r2_val:.4f}")

26/05/30 13:01:56 WARN Instrumentation: [b688a076] regParam is zero, which might cause numerical instability and overfitting.
26/05/30 13:01:57 WARN Instrumentation: [b688a076] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.



[Validation] RMSE: 604.08
[Validation] R2: 0.0690


### Random Forest Baseline

With random forest we can expect clearly better results since it's really built to work with skewed data by nature. As we know the trees will divide the dataset using sub-samples of features and their target values will be averaged to reach a prediction. We are expecting better results for this model with respect to the linear regression. The hyperparameters were fixed as baseline.

In [69]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") 
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_index" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    dropLast=True
)

assembler_inputs = numeric_cols + [f"{c}_ohe" for c in categorical_cols]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="unscaled_features")

scaler = StandardScaler(inputCol="unscaled_features", outputCol="features", withMean=False, withStd=True)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="target_time",
    numTrees=50,                  
    maxDepth=7,                    
    maxBins=64,  
    featureSubsetStrategy="sqrt",
    subsamplingRate=0.8,
    minInstancesPerNode=10,       
    seed=42
)

pipeline_rf = Pipeline(stages=[*indexers, encoder, assembler, scaler, rf])


model_rf = pipeline_rf.fit(df)

predictions_rf = model_rf.transform(df_val)


evaluator_rf = RegressionEvaluator(labelCol="target_time", predictionCol="prediction")

rmse_rf = evaluator_rf.setMetricName("rmse").evaluate(predictions_rf)
print(f"\n[Validation] Random Forest RMSE: {rmse_rf:.2f}")

r2_rf = evaluator_rf.setMetricName("r2").evaluate(predictions_rf)
print(f"[Validation] Random Forest R2: {r2_rf:.4f}")


[Validation] Random Forest RMSE: 435.88
[Validation] Random Forest R2: 0.5153


### MLP Baseline

Before training the neural network, we need to transform and export our data into a format that can be easily loaded by PyTorch. 

We isolate the preprocessing steps into a standalone `feature_pipeline`. This pipeline handles the `StringIndexer`, `OneHotEncoder`, `VectorAssembler`, and `StandardScaler` exactly as done for the baseline models, mapping all raw variables into a single consolidated `features` column.

Since PyTorch cannot natively train on a live, distributed Spark DataFrame in the same way Spark MLlib models do, we serialize the preprocessed training and validation subsets:
* The pipeline is fitted on the training data (`df`).
* Both datasets are transformed, selecting only the processed `features` vector and the `target_time` label.
* The structured data is saved to disk in Parquet format (`TRAIN_PATH` and `VAL_PATH`), which allows efficient chunk-by-chunk loading during the deep learning training loop.

Finally, we programmatically determine the exact size of the final feature vector (`INPUT_DIM`) by inspecting a single sample row. This prevents hardcoding the input layer size of the neural network, automatically adapting to the total number of dimensions generated by the One-Hot Encoding process.

Since it's a very small network (and so not the best architecture for tabular data) we will expect a result in the middle between the 2 previous models.

In [70]:
categorical_cols = ["RatecodeID", "area_pickup", "area_dropoff", "pickup_hour"]
numeric_cols     = ["trip_distance", "tolls_amount", "extra"]
target_col       = "target_time"
 
TRAIN_PATH = "/tmp/mlp_train_features"
VAL_PATH   = "/tmp/mlp_val_features"
WEIGHTS_PATH = "/tmp/mlp_weights.pt"

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]
 
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    dropLast=True
)
 
numeric_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="numeric_raw"
)
 
scaler = StandardScaler(
    inputCol="numeric_raw",
    outputCol="numeric_scaled",
    withMean=False,
    withStd=True
)
 
final_assembler = VectorAssembler(
    inputCols=["numeric_scaled"] + [f"{c}_ohe" for c in categorical_cols],
    outputCol="features"
)
 
feature_pipeline = Pipeline(stages=[
    *indexers,
    encoder,
    numeric_assembler,
    scaler,
    final_assembler
])
 
feature_model = feature_pipeline.fit(df)
 
(
    feature_model
    .transform(df)
    .select("features", target_col)
    .write.mode("overwrite")
    .parquet(TRAIN_PATH)
)
 
(
    feature_model
    .transform(df_val)
    .select("features", target_col)
    .write.mode("overwrite")
    .parquet(VAL_PATH)
)
 
sample      = feature_model.transform(df).select("features").limit(1).collect()
INPUT_DIM   = len(sample[0]["features"])

In [71]:
class TabularNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
 
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
            
        out = self.net(x)
        return out.squeeze(-1) if out.shape[0] > 1 else out.squeeze()

To bridge the gap between Spark's distributed data storage and PyTorch's native tensor-based training, we implement a centralized training utility function. This function handles data ingestion, format conversion, and the optimization loop.


The training routine is divided into three distinct operational phases:

PySpark outputs vector columns in a specialized structure that handles both **dense** and **sparse** formats to save space. The internal helper function reads the serialized Parquet files and dynamically parses these structures:
* **Sparse Vectors:** Reconstructs full 1D arrays by mapping specific values back to their target indices against the overall feature dimension size.
* **Dense Vectors:** Extracts and converts native value arrays directly.
* Both features and targets are reshaped into 2D matrices (`[N, INPUT_DIM]` and `[N, 1]` respectively) to prevent shape mismatch and unintended broadcasting operations during loss computation.

The training loop utilizes advanced optimization techniques to ensure stable convergence:
* **Optimizer:** `AdamW` is employed to leverage adaptive learning rates combined with decoupled weight decay for proper regularization.
* **Learning Rate Scheduler:** `ReduceLROnPlateau` dynamically scales down the learning rate by a factor of 0.5 if the validation loss plateaus for 3 consecutive epochs, allowing for fine-grained convergence.

For each epoch, the function switches between `model.train()` and `model.eval()` modes to appropriately handle regularization layers. It computes the Root Mean Squared Error (RMSE) for both splits, updates the learning rate based on validation performance, and implements a **checkpointing mechanism** that saves only the best model weights (`.pt`) to disk whenever validation loss reaches a new global minimum.

In [72]:
def train_fn(
    train_path,
    val_path,
    weights_path,
    input_dim,
    target_col,
    epochs=50,
    batch_size=512,
    lr=1e-3
):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    import numpy as np
    import pandas as pd

    def parquet_to_tensors(path):
        pdf = pd.read_parquet(path)
        
        X_list = []
        for row in pdf["features"]:
            if isinstance(row, dict) and "type" in row:
                size = int(row["size"])
                if row["type"] == 0 or "indices" in row: 
                    indices = row["indices"]
                    values = row["values"]
                    vec = np.zeros(size, dtype=np.float32)
                    vec[indices] = values
                    X_list.append(vec)
                else:
                    X_list.append(np.array(row["values"], dtype=np.float32))
            elif isinstance(row, dict) and "values" in row:
                X_list.append(np.array(row["values"], dtype=np.float32))
            else:
                X_list.append(np.array(row, dtype=np.float32))
        
        X = np.array(X_list, dtype=np.float32)
        y = pdf[target_col].values.astype(np.float32)
        
        return torch.from_numpy(X), torch.from_numpy(y)

    X_train, y_train = parquet_to_tensors(train_path)
    X_val,   y_val   = parquet_to_tensors(val_path)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )

    model = TabularNet(input_dim)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)

    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            opt.zero_grad()
            preds = model(xb).squeeze()
            loss = F.mse_loss(preds, yb)
            loss.backward()
            opt.step()
            train_loss += loss.item() * len(xb)
        train_rmse = (train_loss / (len(train_loader) * batch_size)) ** 0.5

        model.eval()
        with torch.no_grad():
            val_preds = model(X_val).squeeze()
            val_loss  = F.mse_loss(val_preds, y_val).item()
        val_rmse = val_loss ** 0.5

        scheduler.step(val_loss)
        print(f"Epoch {epoch+1:>3}/{epochs} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), weights_path)

    print(f"\nBest Val RMSE: {best_val_loss**0.5:.4f}")
    return best_val_loss ** 0.5

In [73]:
best_rmse = TorchDistributor(
    num_processes=1,   
    local_mode=True,   
    use_gpu=False
).run(
    train_fn,
    TRAIN_PATH,
    VAL_PATH,
    WEIGHTS_PATH,
    INPUT_DIM,
    target_col
)
 
print(f"\nMLP Val RMSE: {best_rmse:.4f}")

Started local training with 1 processes


Epoch   1/50 | Train RMSE: 927.4504 | Val RMSE: 863.7220
Epoch   2/50 | Train RMSE: 543.4413 | Val RMSE: 581.2742
Epoch   3/50 | Train RMSE: 398.0193 | Val RMSE: 560.2807
Epoch   4/50 | Train RMSE: 385.8836 | Val RMSE: 555.1573
Epoch   5/50 | Train RMSE: 378.8423 | Val RMSE: 549.5922
Epoch   6/50 | Train RMSE: 373.0201 | Val RMSE: 546.3603
Epoch   7/50 | Train RMSE: 369.4728 | Val RMSE: 545.9116
Epoch   8/50 | Train RMSE: 367.8209 | Val RMSE: 541.1451
Epoch   9/50 | Train RMSE: 365.7057 | Val RMSE: 537.9370
Epoch  10/50 | Train RMSE: 364.0072 | Val RMSE: 538.1925
Epoch  11/50 | Train RMSE: 362.6479 | Val RMSE: 532.9732
Epoch  12/50 | Train RMSE: 360.8504 | Val RMSE: 536.5407
Epoch  13/50 | Train RMSE: 360.3693 | Val RMSE: 533.8927
Epoch  14/50 | Train RMSE: 359.3234 | Val RMSE: 534.4512
Epoch  15/50 | Train RMSE: 359.2608 | Val RMSE: 536.4882
Epoch  16/50 | Train RMSE: 356.5907 | Val RMSE: 532.3892
Epoch  17/50 | Train RMSE: 356.4475 | Val RMSE: 530.0638
Epoch  18/50 | Train RMSE: 354.

Finished local training with 1 processes



MLP Val RMSE: 526.7124


### Hyperparameters for Random Forest

As predicted our best model as baseline was the random forest, we will make a random search on our hyperparameter grid to find a better solution

In [74]:
SEED             = 42
N_COMBINATIONS   = 10
 
random.seed(SEED)

In [75]:
param_space = {
    "numTrees":              [50, 100, 150, 200],
    "maxDepth":              [4, 6, 8, 10, 12],
    "featureSubsetStrategy": ["sqrt", "log2", "onethird"],
    "subsamplingRate":       [0.6, 0.7, 0.8, 1.0],
    "minInstancesPerNode":   [1, 5, 10, 20],
    "maxBins":               [32, 64, 128],
}
 
def sample_params(space, seed):
    random.seed(seed)
    return {k: random.choice(v) for k, v in space.items()}
 
combinations = [sample_params(param_space, SEED + i) for i in range(N_COMBINATIONS)]

In [76]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]
 
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    dropLast=True
)
 
assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{c}_ohe" for c in categorical_cols],
    outputCol="features"
)

Instead of fitting the preprocessing steps multiple times, we execute the `feature_pipeline` as a single processing block. Crucially, we apply the `.persist()` operator to the resulting transformed DataFrames (`df_train_feat` and `df_val_feat`). 

Because Spark uses lazy evaluation, caching these preprocessed features in memory prevents the cluster from re-running the entire string indexing, one-hot encoding, and scaling pipeline every time a different model accesses the data. This drastically cuts down total execution time.

In [77]:
feature_pipeline = Pipeline(stages=[*indexers, encoder, assembler])
feature_model    = feature_pipeline.fit(df)
 

df_train_feat = feature_model.transform(df).select("features", target_col).persist()
df_val_feat   = feature_model.transform(df_val).select("features", target_col).persist()

rmse_evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
r2_evaluator   = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")

In [78]:
results = []
best_rmse = float("inf")
best_params = None



for i, params in enumerate(combinations):

 
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol=target_col,
        seed=SEED,
        **params
    )
 
    model = rf.fit(df_train_feat)
    val_pred = model.transform(df_val_feat)
    
    rmse = rmse_evaluator.evaluate(val_pred)
    r2   = r2_evaluator.evaluate(val_pred)
 
    results.append({"params": params, "rmse": rmse, "r2": r2})
    print(f"    -> [Results] RMSE: {rmse:.4f} | R2: {r2:.4f}")
    
    if rmse < best_rmse:
        best_rmse = rmse
        best_params = params

    -> [Results] RMSE: 451.3337 | R2: 0.4803


26/05/30 13:05:59 WARN DAGScheduler: Broadcasting large task binary with size 1345.6 KiB


    -> [Results] RMSE: 406.7428 | R2: 0.5779


26/05/30 13:06:40 WARN DAGScheduler: Broadcasting large task binary with size 1332.6 KiB
26/05/30 13:06:54 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/05/30 13:07:12 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB
26/05/30 13:07:37 WARN DAGScheduler: Broadcasting large task binary with size 6.0 MiB
26/05/30 13:08:04 WARN DAGScheduler: Broadcasting large task binary with size 1189.5 KiB
26/05/30 13:08:15 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB
26/05/30 13:09:00 WARN DAGScheduler: Broadcasting large task binary with size 1687.9 KiB
26/05/30 13:09:16 WARN DAGScheduler: Broadcasting large task binary with size 13.3 MiB
26/05/30 13:10:09 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/05/30 13:10:29 WARN DAGScheduler: Broadcasting large task binary with size 18.7 MiB
26/05/30 13:11:40 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
26/05/30 13:12:06 WARN DAGScheduler: Broadc

    -> [Results] RMSE: 388.2627 | R2: 0.6154


...

    -> [Results] RMSE: 440.6419 | R2: 0.5046


26/05/30 13:19:42 WARN DAGScheduler: Broadcasting large task binary with size 1429.3 KiB
26/05/30 13:19:54 WARN DAGScheduler: Broadcasting large task binary with size 1988.2 KiB
26/05/30 13:20:08 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB


    -> [Results] RMSE: 438.8632 | R2: 0.5086


26/05/30 13:20:54 WARN DAGScheduler: Broadcasting large task binary with size 1248.7 KiB
26/05/30 13:21:04 WARN DAGScheduler: Broadcasting large task binary with size 2025.5 KiB
26/05/30 13:21:17 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/05/30 13:21:34 WARN DAGScheduler: Broadcasting large task binary with size 4.7 MiB
26/05/30 13:21:56 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
26/05/30 13:22:17 WARN DAGScheduler: Broadcasting large task binary with size 1147.2 KiB
26/05/30 13:22:26 WARN DAGScheduler: Broadcasting large task binary with size 9.5 MiB
26/05/30 13:22:59 WARN DAGScheduler: Broadcasting large task binary with size 1495.2 KiB
26/05/30 13:23:10 WARN DAGScheduler: Broadcasting large task binary with size 12.9 MiB
26/05/30 13:23:50 WARN DAGScheduler: Broadcasting large task binary with size 1880.7 KiB


    -> [Results] RMSE: 402.7298 | R2: 0.5862


In [ ]:
df_train_feat.unpersist()
df_val_feat.unpersist()

print(f"Best model")
print(f"Best Validation RMSE: {best_rmse:.4f}")
print(f"Optimal parameters: {best_params}")

Best model
Best Validation RMSE: 388.2627
Optimal parameters: {'numTrees': 200, 'maxDepth': 12, 'featureSubsetStrategy': 'onethird', 'subsamplingRate': 0.6, 'minInstancesPerNode': 5, 'maxBins': 64}


## Models for Fare amount

The process employed for the fare follows the exact same process as the one done for the time, we will avoid useless explanations. 

After checking our data we noticed some very suspect values that were not present on the dataset for the time. We decided to make a subsequent cap on the acceptable range of the fare

In [80]:
categorical_cols = ["RatecodeID", "area_pickup", "area_dropoff", "pickup_hour"]
numeric_cols     = ["trip_distance", "tolls_amount", "extra"]
target_col       = "fare_amount"

df     = df.filter(F.col("fare_amount") <= 50)
df_val = df_val.filter(F.col("fare_amount") <= 50)

### Linear regression Baseline

In [81]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep")
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_index" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    dropLast=True
)

numeric_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="numeric_vec"
)

scaler = StandardScaler(
    inputCol="numeric_vec",
    outputCol="numeric_scaled",
    withMean=False, 
    withStd=True
)

final_assembler = VectorAssembler(
    inputCols=["numeric_scaled"] + [f"{c}_ohe" for c in categorical_cols],
    outputCol="features"
)

lr = LinearRegression(
    featuresCol="features",
    labelCol=target_col
)


pipeline = Pipeline(stages=[
    *indexers,
    encoder,
    numeric_assembler,
    scaler,
    final_assembler,
    lr
])


model = pipeline.fit(df)


train_pred = model.transform(df)
val_pred   = model.transform(df_val)



evaluator = RegressionEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="rmse"
)

rmse_val = evaluator.evaluate(val_pred)
print(f"\n[Validation] RMSE: {rmse_val:.2f}")

r2_val = evaluator.setMetricName("r2").evaluate(val_pred)
print(f"[Validation] R2: {r2_val:.4f}")

26/05/30 13:24:28 WARN Instrumentation: [a507e1df] regParam is zero, which might cause numerical instability and overfitting.
26/05/30 13:24:29 WARN Instrumentation: [a507e1df] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.



[Validation] RMSE: 6.25
[Validation] R2: 0.3090


### Random forest Baseline

In [82]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") 
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_index" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    dropLast=True
)

assembler_inputs = numeric_cols + [f"{c}_ohe" for c in categorical_cols]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="unscaled_features")

scaler = StandardScaler(inputCol="unscaled_features", outputCol="features", withMean=False, withStd=True)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=50,                  
    maxDepth=7,                    
    maxBins=64,  
    featureSubsetStrategy="sqrt",
    subsamplingRate=0.8,
    minInstancesPerNode=10,       
    seed=42
)

pipeline_rf = Pipeline(stages=[*indexers, encoder, assembler, scaler, rf])

model_rf = pipeline_rf.fit(df)

predictions_rf = model_rf.transform(df_val)


evaluator_rf = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")

rmse_rf = evaluator_rf.setMetricName("rmse").evaluate(predictions_rf)
print(f"\n[Validation] Random Forest RMSE: {rmse_rf:.2f}")

r2_rf = evaluator_rf.setMetricName("r2").evaluate(predictions_rf)
print(f"[Validation] Random Forest R2: {r2_rf:.4f}")


[Validation] Random Forest RMSE: 4.34
[Validation] Random Forest R2: 0.6657


### MLP Baseline

In [83]:
categorical_cols = ["RatecodeID", "area_pickup", "area_dropoff", "pickup_hour"]
numeric_cols     = ["trip_distance", "tolls_amount", "extra"]
target_col       = "fare_amount"
 
TRAIN_PATH = "/tmp/mlp_train_features"
VAL_PATH   = "/tmp/mlp_val_features"
WEIGHTS_PATH = "/tmp/mlp_weights.pt"


indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]
 
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    dropLast=True
)
 
numeric_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="numeric_raw"
)
 
scaler = StandardScaler(
    inputCol="numeric_raw",
    outputCol="numeric_scaled",
    withMean=False,
    withStd=True
)
 
final_assembler = VectorAssembler(
    inputCols=["numeric_scaled"] + [f"{c}_ohe" for c in categorical_cols],
    outputCol="features"
)
 
feature_pipeline = Pipeline(stages=[
    *indexers,
    encoder,
    numeric_assembler,
    scaler,
    final_assembler
])
 
feature_model = feature_pipeline.fit(df)
 
(
    feature_model
    .transform(df)
    .select("features", target_col)
    .write.mode("overwrite")
    .parquet(TRAIN_PATH)
)
 
(
    feature_model
    .transform(df_val)
    .select("features", target_col)
    .write.mode("overwrite")
    .parquet(VAL_PATH)
)
 

sample      = feature_model.transform(df).select("features").limit(1).collect()
INPUT_DIM   = len(sample[0]["features"])


In [84]:
best_rmse = TorchDistributor(
    num_processes=1,   
    local_mode=True,   
    use_gpu=False
).run(
    train_fn,
    TRAIN_PATH,
    VAL_PATH,
    WEIGHTS_PATH,
    INPUT_DIM,
    target_col
)
 
print(f"\nMLP Val RMSE: {best_rmse:.4f}")

Started local training with 1 processes


Epoch   1/50 | Train RMSE: 6.1948 | Val RMSE: 6.3764
Epoch   2/50 | Train RMSE: 3.7265 | Val RMSE: 6.1879
Epoch   3/50 | Train RMSE: 3.5190 | Val RMSE: 5.9402
Epoch   4/50 | Train RMSE: 3.4026 | Val RMSE: 5.8806
Epoch   5/50 | Train RMSE: 3.3483 | Val RMSE: 5.8105
Epoch   6/50 | Train RMSE: 3.2499 | Val RMSE: 5.7097
Epoch   7/50 | Train RMSE: 3.2082 | Val RMSE: 5.7564
Epoch   8/50 | Train RMSE: 3.1483 | Val RMSE: 5.7891
Epoch   9/50 | Train RMSE: 3.1250 | Val RMSE: 5.6117
Epoch  10/50 | Train RMSE: 3.0745 | Val RMSE: 5.5332
Epoch  11/50 | Train RMSE: 3.0593 | Val RMSE: 5.6402
Epoch  12/50 | Train RMSE: 3.0545 | Val RMSE: 5.5634
Epoch  13/50 | Train RMSE: 3.0206 | Val RMSE: 5.4295
Epoch  14/50 | Train RMSE: 2.9861 | Val RMSE: 5.5722
Epoch  15/50 | Train RMSE: 2.9630 | Val RMSE: 5.4561
Epoch  16/50 | Train RMSE: 2.9467 | Val RMSE: 5.4027
Epoch  17/50 | Train RMSE: 2.8930 | Val RMSE: 5.4788
Epoch  18/50 | Train RMSE: 2.8596 | Val RMSE: 5.3348
Epoch  19/50 | Train RMSE: 2.8519 | Val RMSE: 

Finished local training with 1 processes



MLP Val RMSE: 5.2456


### Hyperparameters for Random Forest

In [85]:
SEED             = 67
N_COMBINATIONS   = 10
 
random.seed(SEED)

In [86]:
param_space = {
    "numTrees":              [50, 100, 150, 200],
    "maxDepth":              [4, 6, 8, 10, 12],
    "featureSubsetStrategy": ["sqrt", "log2", "onethird"],
    "subsamplingRate":       [0.6, 0.7, 0.8, 1.0],
    "minInstancesPerNode":   [1, 5, 10, 20],
    "maxBins":               [32, 64, 128],
}
 
def sample_params(space, seed):
    random.seed(seed)
    return {k: random.choice(v) for k, v in space.items()}
 
combinations = [sample_params(param_space, SEED + i) for i in range(N_COMBINATIONS)]

In [87]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]
 
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    dropLast=True
)
 
assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{c}_ohe" for c in categorical_cols],
    outputCol="features"
)

In [88]:
feature_pipeline = Pipeline(stages=[*indexers, encoder, assembler])
feature_model    = feature_pipeline.fit(df)
 
df_train_feat = feature_model.transform(df).select("features", target_col).persist()
df_val_feat   = feature_model.transform(df_val).select("features", target_col).persist()
 
rmse_evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
r2_evaluator   = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")

In [89]:
results = []
best_rmse = float("inf")
best_params = None


for i, params in enumerate(combinations):
    print(f"\n[{i+1}/{N_COMBINATIONS}] Configurazione in esecuzione...")
 
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol=target_col,
        seed=SEED,
        **params
    )
 
    model = rf.fit(df_train_feat)
    val_pred = model.transform(df_val_feat)
    

    rmse = rmse_evaluator.evaluate(val_pred)
    r2   = r2_evaluator.evaluate(val_pred)
 
    results.append({"params": params, "rmse": rmse, "r2": r2})
    print(f"    -> [Results] RMSE: {rmse:.4f} | R2: {r2:.4f}")

    if rmse < best_rmse:
        best_rmse = rmse
        best_params = params


[1/10] Configurazione in esecuzione...


    -> [Results] RMSE: 5.2993 | R2: 0.5025

[2/10] Configurazione in esecuzione...


26/05/30 13:28:16 WARN DAGScheduler: Broadcasting large task binary with size 1470.0 KiB
26/05/30 13:28:28 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/05/30 13:28:43 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/05/30 13:29:00 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB
26/05/30 13:29:21 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
26/05/30 13:29:44 WARN DAGScheduler: Broadcasting large task binary with size 6.0 MiB
26/05/30 13:30:11 WARN DAGScheduler: Broadcasting large task binary with size 7.3 MiB


    -> [Results] RMSE: 3.6097 | R2: 0.7692

[3/10] Configurazione in esecuzione...


    -> [Results] RMSE: 5.0943 | R2: 0.5402

[4/10] Configurazione in esecuzione...


...

    -> [Results] RMSE: 4.2688 | R2: 0.6772

[9/10] Configurazione in esecuzione...


26/05/30 13:39:43 WARN DAGScheduler: Broadcasting large task binary with size 1116.9 KiB
26/05/30 13:39:54 WARN DAGScheduler: Broadcasting large task binary with size 1817.2 KiB
26/05/30 13:40:07 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/05/30 13:40:24 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/05/30 13:40:46 WARN DAGScheduler: Broadcasting large task binary with size 5.9 MiB
26/05/30 13:41:13 WARN DAGScheduler: Broadcasting large task binary with size 8.0 MiB
26/05/30 13:41:44 WARN DAGScheduler: Broadcasting large task binary with size 1164.8 KiB
26/05/30 13:41:53 WARN DAGScheduler: Broadcasting large task binary with size 10.6 MiB
26/05/30 13:42:28 WARN DAGScheduler: Broadcasting large task binary with size 1396.7 KiB
26/05/30 13:42:38 WARN DAGScheduler: Broadcasting large task binary with size 13.7 MiB
26/05/30 13:43:23 WARN DAGScheduler: Broadcasting large task binary with size 1634.6 KiB


    -> [Results] RMSE: 3.7936 | R2: 0.7450

[10/10] Configurazione in esecuzione...


26/05/30 13:44:09 WARN DAGScheduler: Broadcasting large task binary with size 1165.5 KiB
26/05/30 13:44:19 WARN DAGScheduler: Broadcasting large task binary with size 1647.9 KiB
26/05/30 13:44:31 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/05/30 13:44:45 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/05/30 13:45:01 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB


    -> [Results] RMSE: 4.1558 | R2: 0.6940


In [90]:
df_train_feat.unpersist()
df_val_feat.unpersist()

print(f"Best model")

print(f"Best Validation RMSE: {best_rmse:.4f}")
print(f"Best params: {best_params}")

Best model
Best Validation RMSE: 2.9084
Best params: {'numTrees': 50, 'maxDepth': 12, 'featureSubsetStrategy': 'onethird', 'subsamplingRate': 0.7, 'minInstancesPerNode': 10, 'maxBins': 128}


## Test for time

Now that we have our results we can test our models on our test set. We will retrain the models with their new hyperparameters on the train and on the validation set put together and, we will test them on the test set (that we already prepared and cleaned)

In [91]:
df_fin = df.unionByName(df_val)
df_fin = df_fin.persist()

df_fin = df_fin.filter(F.col("fare_amount") <= 50)
df_test = df_test.filter(F.col("fare_amount") <= 50)

In [92]:
categorical_cols = ["RatecodeID", "area_pickup", "area_dropoff", "pickup_hour"]
numeric_cols = ["trip_distance", "tolls_amount", "extra"]


target_col = "target_time"

In [93]:
indexers = [StringIndexer(inputCol=c, outputCol=c+"_index", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCols=[c+"_index"], outputCols=[c+"_ohe"]) for c in categorical_cols]


assembler_inputs = numeric_cols + [c+"_ohe" for c in categorical_cols]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="unscaled_features")

scaler = StandardScaler(inputCol="unscaled_features", outputCol="features")

rf_time = RandomForestRegressor(
    featuresCol="features",
    labelCol="target_time",
    numTrees=200,                  
    maxDepth=12,                  
    maxBins=64,                   
    featureSubsetStrategy="onethird",
    subsamplingRate=0.6,
    minInstancesPerNode=5,       
    seed=42
)


pipeline_rf_time = Pipeline(stages=indexers + encoders + [assembler, scaler, rf_time])
model_rf_time = pipeline_rf_time.fit(df_fin)
predictions_rf_time = model_rf_time.transform(df_test)


evaluator_rf_time = RegressionEvaluator(labelCol="target_time", predictionCol="prediction", metricName="rmse")
rmse_rf_time = evaluator_rf_time.evaluate(predictions_rf_time)

print(f"Random Forest RMSE: {rmse_rf_time:.2f}")

r2_rf_time = evaluator_rf_time.setMetricName("r2").evaluate(predictions_rf_time)
print(f"Random Forest R2: {r2_rf_time:.4f}")

26/05/30 13:46:33 WARN DAGScheduler: Broadcasting large task binary with size 1337.4 KiB
26/05/30 13:46:55 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/05/30 13:47:24 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB
26/05/30 13:48:05 WARN DAGScheduler: Broadcasting large task binary with size 5.9 MiB
26/05/30 13:48:43 WARN DAGScheduler: Broadcasting large task binary with size 1257.6 KiB
26/05/30 13:48:59 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB
26/05/30 13:50:08 WARN DAGScheduler: Broadcasting large task binary with size 1815.7 KiB
26/05/30 13:50:33 WARN DAGScheduler: Broadcasting large task binary with size 13.4 MiB
26/05/30 13:52:05 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/05/30 13:52:40 WARN DAGScheduler: Broadcasting large task binary with size 19.0 MiB
26/05/30 13:54:48 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/05/30 13:55:32 WARN DAGScheduler: Broadc

Random Forest RMSE: 318.48


Random Forest R2: 0.6647


The model demonstrates strong predictive capabilities for estimating target_time. With an R2 score of 0.6647, the model successfully accounts for approximately 66.5% of the total variance in trip durations. Given the inherent real-world volatility of transit and traffic data, achieving an R2 well above 0.60 indicates a highly robust, reliable fit that captures major underlying traffic patterns without overfitting.

The model was trained using a tailored configuration designed to optimize stability, split diversity, and execution performance on large-scale data:

| Hyperparameter | Value | Strategic Purpose |
| :--- | :--- | :--- |
| numTrees | 200 | Provides a large, stable ensemble that reduces variance and mitigates over-prediction. |
| maxDepth | 12 | Allows the model to capture deep, complex non-linear interactions without risking extreme overfitting. |
| maxBins | 64 | Increases continuous variable split granularity, preserving precision for geographical and temporal features. |
| featureSubsetStrategy | "onethird" | Decouples individual tree correlations by choosing a random 1/3 subset of features per split, enhancing ensemble diversity. |
| subsamplingRate | 0.6 | Introduces bagging noise (using 60% of data per tree), which boosts generalization performance and speeds up execution. |
| minInstancesPerNode | 5 | Acts as a regularizer, stopping splits that isolate fewer than 5 samples to prevent memorizing outliers. |
| seed | 42 | Ensures deterministic execution and exact experiment reproducibility. |

Model evaluation yielded a Root Mean Squared Error (RMSE) of 318.48 seconds, which is approximately 5.3 minutes, alongside a Coefficient of Determination (R2) of 0.6647.

An RMSE of roughly 5.3 minutes is remarkably good for urban transit modeling. It proves that the model effectively maps out geographic hotspot distances, hour-of-day traffic waves, and fee metrics to estimate arrival windows accurately. Additionally, the combination of a 0.6 subsampling rate and a one-third feature subset strategy successfully guarded the model against overfitting, translating into a strong, generalized performance profile on unseen evaluation data.

## Test for Fare

In [94]:
categorical_cols = ["RatecodeID", "area_pickup", "area_dropoff", "pickup_hour"]
numeric_cols = ["trip_distance", "tolls_amount", "extra"]


target_col = "fare_amount"

In [95]:
indexers = [StringIndexer(inputCol=c, outputCol=c+"_index", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCols=[c+"_index"], outputCols=[c+"_ohe"]) for c in categorical_cols]


assembler_inputs = numeric_cols + [c+"_ohe" for c in categorical_cols]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="unscaled_features")

scaler = StandardScaler(inputCol="unscaled_features", outputCol="features")

rf_fare = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=50,                 
    maxDepth=12,                   
    maxBins=128,                   
    featureSubsetStrategy="onethird",
    subsamplingRate=0.7,
    minInstancesPerNode=10,       
    seed=42
)

pipeline_rf_fare = Pipeline(stages=indexers + encoders + [assembler, scaler, rf_fare])
model_rf_fare = pipeline_rf_fare.fit(df_fin)
predictions_rf_fare = model_rf_fare.transform(df_test)

evaluator_rf_fare = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse")
rmse_rf_fare = evaluator_rf_fare.evaluate(predictions_rf_fare)

print(f"Random Forest RMSE: {rmse_rf_fare:.2f}")

r2_rf_fare = evaluator_rf_fare.setMetricName("r2").evaluate(predictions_rf_fare)
print(f"Random Forest R2: {r2_rf_fare:.4f}")

26/05/30 14:00:59 WARN DAGScheduler: Broadcasting large task binary with size 1445.5 KiB
26/05/30 14:01:17 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/05/30 14:01:39 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/05/30 14:02:06 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
26/05/30 14:02:38 WARN DAGScheduler: Broadcasting large task binary with size 5.2 MiB


Random Forest RMSE: 2.56


Random Forest R2: 0.8896


The model demonstrates excellent predictive capabilities for estimating fare_amount. With an R2 score of 0.8896, the model successfully accounts for approximately 89.0% of the total variance in trip fares. This indicates a highly accurate and dependable fit, meaning the engineered features strongly explain the underlying pricing structure.

The model was trained using a tailored configuration designed to optimize split precision, robustness against noise, and execution speed on large-scale financial data:

| Hyperparameter | Value | Strategic Purpose |
| :--- | :--- | :--- |
| numTrees | 50 | Balances ensemble variance reduction with fast execution speed for financial metrics. |
| maxDepth | 12 | Standardizes tree depth to capture complex non-linear pricing logic without overfitting. |
| maxBins | 128 | Doubles split granularity to provide high precision for continuous values like trip distance. |
| featureSubsetStrategy | "onethird" | Selects a random 1/3 subset of features per split to decorrelate trees and ensure ensemble diversity. |
| subsamplingRate | 0.7 | Utilizes 70% of the data per tree to introduce bagging noise, boosting generalization and preventing memorization. |
| minInstancesPerNode | 10 | Imposes strict regularization by requiring at least 10 samples per leaf, smoothing out outlier pricing. |
| seed | 42 | Ensures deterministic execution and exact experiment reproducibility. |

Model evaluation yielded a Root Mean Squared Error (RMSE) of 2.56, which represents an average deviation of approximately $2.56, alongside a Coefficient of Determination (R2) of 0.8896.

An RMSE of $2.56 is exceptionally low for urban fare modeling, proving that the model tightly tracks actual trip costs. This high level of precision shows that maximizing the continuous split capacity with a higher maxBins parameter allowed the model to effectively interpret continuous spatial and temporal parameters. Furthermore, the combination of a 0.7 subsampling rate and strict node instance limits effectively insulated the network from fare anomalies, leading to a highly reliable and generalized model on unseen data.

# Streaming

### Manual

The system design focuses on simulating a real-time event processing architecture tailored for a production-grade urban mobility application. The primary business objective of this infrastructure is to minimize user latency by instantly delivering duration and fare estimates (e.g., streaming feedback to a mobile application interface) immediately after a customer requests a ride. 

The implementation simulates a live data stream using a micro-batching mechanism over historical log entries to replicate production streaming behavior safely in a test environment.

| Component | Engineering Implementation | Academic and Practical Rationale |
| :--- | :--- | :--- |
| **Ingestion Simulation** | Sequential row indexing from collected data arrays converted back into isolated DataFrame objects. | Mimics the arrival of an un-batched, isolated payload from an event broker (e.g., Apache Kafka). This enforces streaming-first development constraints, proving the pipeline can handle instantaneous point-in-time processing. |
| **Schema Pruning** | Immediate exclusion of transactional and post-trip features (`payment_type`, `tip_amount`, `total_amount`). | Eliminates target leakage and drops metadata irrelevant to upstream estimators. This optimizes network throughout and payload parsing speeds, which are critical for real-time mobile application feedback. |
| **Temporal Transformation** | Extraction of cyclical integer features from raw timestamps (`pickup_hour`). | Standardizes unstructured time entries into predictable seasonal bins (0–23), allowing the predictive models to adapt to time-of-day traffic bottlenecks. |
| **Geospatial Fallback** | Native mathematical implementation of the Haversine formula for distance estimation. | Solves data quality issues at the edge. If the incoming telemetry registers corrupted or missing odometer distances, the pipeline reconstructs spatial displacement using coordinate arcs, avoiding downstream processing failures. |
| **Geofencing Mechanics** | Loop-unrolling of conditional statements against bounded Euclidean radii. | Emulates a low-latency reverse-geocoding microservice. By avoiding intensive multi-layer spatial spatial-joins, the system resolves location coordinates to logical terminal hubs (e.g., Airports) with minimal CPU cycle overhead. |
| **Statistical Regularization** | Boundary clipping based on distribution parameters derived from offline baseline data. | Guarantees pipeline idempotence and robustness against anomaly propagation. Corrupted inputs are regularized using expected historical distributions without modifying real airport transit premiums. |
| **Pipeline Inference** | Serial deployment of pre-compiled pipeline models (`model_rf_time`, `model_rf_fare`). | Demonstrates a decoupled machine learning architecture. The system executes feature engineering, scaling, and double-label inference in a single pass, immediately printing estimated durations and prices. |

This modular simulation cleanly separates streaming state management from transformation logic. By testing the inference engine under single-row constraints, the design proves that offline training configurations can be directly transferred into online production environments without introducing architectural feature drift, ensuring users receive fast and reliable feedback on their devices.

In [104]:
if 'current_row_index' not in globals():
    global current_row_index
    current_row_index = 0

In [105]:
path_streaming_file = "streaming.csv"
df_streaming = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path_streaming_file)

all_rows = df_streaming.collect()

In [107]:
if current_row_index < len(all_rows):

    single_python_row = all_rows[current_row_index]
    df = spark.createDataFrame([single_python_row], df_streaming.schema)

    df = df.drop(
        "VendorID", "store_and_fwd_flag", "payment_type",
        "passenger_count", "total_amount", "tip_amount",
        "mta_tax", "improvement_surcharge"
    )


    df = df.withColumn("pickup_hour", hour(col("tpep_pickup_datetime")).cast("integer")) \
           .drop("tpep_pickup_datetime")


    EARTH_RADIUS = 6371.0
    lat1 = radians(col("pickup_latitude"))
    lon1 = radians(col("pickup_longitude"))
    lat2 = radians(col("dropoff_latitude"))
    lon2 = radians(col("dropoff_longitude"))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a_expr = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    haversine_dist = EARTH_RADIUS * 2 * atan2(sqrt(a_expr), sqrt(1 - a_expr))

    df = df.withColumn(
        "trip_distance",
        when(
            (col("trip_distance") <= 0) | col("trip_distance").isNull(),
            haversine_dist
        ).otherwise(col("trip_distance"))
    )


    pickup_area_expr = F.lit("Other")
    dropoff_area_expr = F.lit("Other")

    for area_name, (h_lon, h_lat) in NYC_HOTSPOTS.items():

        in_pickup_zone  = (pow(col("pickup_longitude")  - h_lon, 2) + pow(col("pickup_latitude")  - h_lat, 2)) < 0.05**2
        in_dropoff_zone = (pow(col("dropoff_longitude") - h_lon, 2) + pow(col("dropoff_latitude") - h_lat, 2)) < 0.05**2
        pickup_area_expr  = when(in_pickup_zone,  F.lit(area_name)).otherwise(pickup_area_expr)
        dropoff_area_expr = when(in_dropoff_zone, F.lit(area_name)).otherwise(dropoff_area_expr)

    df = df.withColumn("area_pickup",  pickup_area_expr) \
           .withColumn("area_dropoff", dropoff_area_expr) \
           .drop("pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude")

    df = df.filter(col("tolls_amount") <= 50) \
           .withColumn("extra", F.abs(col("extra")))

    df = df.withColumn(
        "is_airport_trip",
        col("RatecodeID").isin(2, 3) |
        col("area_pickup").isin("JFK_Airport", "LaGuardia_Airport") |
        col("area_dropoff").isin("JFK_Airport", "LaGuardia_Airport")
    )


    for col_name, params in learned_parameters.items():
        if col_name in df.columns:
            df = df.withColumn(
                col_name,
                when(col("is_airport_trip"), col(col_name))
                .when(col(col_name) > params["upper_bound"], F.lit(float(params["mean"])))
                .otherwise(col(col_name))
            )


    cols_to_drop = (
        [c + "_index" for c in categorical_cols] +
        [c + "_ohe"   for c in categorical_cols] +
        ["unscaled_features", "features"]
    )

    df = model_rf_time.transform(df) \
                      .withColumnRenamed("prediction", "predicted_time") \
                      .drop(*cols_to_drop)

    df = model_rf_fare.transform(df) \
                      .withColumnRenamed("prediction", "predicted_fare")

    final_result = df.collect()[0]

    print("\n" + "[EVENT] Incoming Live Ride Request")
    print(f"Request ID: TR-{current_row_index + 1:04d} | Stream Index: {current_row_index}")
    print("-" * 50)
    print(f"  Route:       {final_result['area_pickup']} -> {final_result['area_dropoff']}")
    print(f"  Schedule:    {final_result['pickup_hour']:02d}:00")
    print(f"  Distance:    {final_result['trip_distance']:.2f} km")
    print("-" * 50)
    print(f"  Est. Duration:  {final_result['predicted_time']:.1f} secs")
    print(f"  Est. Fare:      ${final_result['predicted_fare']:.2f}")
    print("-" * 50 + "\n")

    current_row_index += 1

else:
    print("\n[INFO] Stream ended. All rows processed. Reset with: current_row_index = 0\n")


[EVENT] Incoming Live Ride Request
Request ID: TR-0003 | Stream Index: 2
--------------------------------------------------
  Route:       Harlem -> LaGuardia_Airport
  Schedule:    05:00
  Distance:    9.20 km
--------------------------------------------------
  Est. Duration:  1373.6 secs
  Est. Fare:      $29.74
--------------------------------------------------



### Automatic

This architectural refinement replaces the previous driver-side sequential indexing loop with an asynchronous, distributed file-stream listener using Spark Structured Streaming. This design isolates data source ingestion from transformation handling, allowing the platform to dynamically monitor storage layers and process incoming events asynchronously.

| Structural Layer | Technical Implementation | Academic and Practical Rationale |
| :--- | :--- | :--- |
| **Directory-Based Streaming Source** | `spark.readStream.format("csv")` targeting an automated local directory with explicit structural constraints. | Simulates an edge-node landing zone or file system storage sink. This setup establishes an operational data provider that watches a specific target directory for new data entries asynchronously. |
| **Ingestion Micro-Batch Choking** | `.option("maxFilesPerTrigger", 1)` paired with an explicit processing trigger interval of `10 seconds`. | Enforces strict backpressure controls at the streaming boundary. This ensures that the engine pulls precisely one file per micro-batch window, preventing sudden volume spikes from exhausting driver memory and guaranteeing constant-time execution. |
| **Strict Schema Constraints** | Explicit compilation of a native `StructType` template (`taxi_schema`). | Eliminates type-inference latency overhead during stream validation. This safeguards the transformation layer against structural mutations, missing parameters, or incompatible data types from corrupted payloads. |
| **Batch Interface Encapsulation** | Custom `.foreachBatch(process_batch)` orchestration method wrapper. | Decouples streaming metadata handles from standard relational logic. This design allows you to reuse pre-trained batch ML pipelines inside a live streaming context, ensuring consistency without rewriting the underlying prediction logic. |
| **Zero-Value Boundary Gates** | Early short-circuit filter check using an explicit evaluation gate: `if batch_df.count() == 0: return`. | Enhances engine efficiency during periods of low traffic. It prevents the system from spinning up empty task runners and passing null frames to the ML model pipelines when no new ride data files are detected. |
| **Distributed Task Output** | Internal dataset extraction using micro-batch collectors (`df.collect()`). | Emulates an application push event trigger. In an enterprise system, this step would publish parsed JSON payloads directly to an external notification database or real-time message bus to update user screens instantly. |

This file-source streaming design bridges the gap between local prototype loops and production messaging networks. By swapping out the file target path for an enterprise-level data connector, this exact processing logic can be migrated directly to production clusters to deliver instant, low-latency fare feedback to active mobile users.

In [10]:
TAXI_STREAM_DIR = "taxi_stream"
os.makedirs(TAXI_STREAM_DIR, exist_ok=True)

path_streaming_file = "streaming.csv"
df_streaming = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path_streaming_file)
all_rows = df_streaming.collect()

for i, row in enumerate(all_rows):
    spark.createDataFrame([row], df_streaming.schema) \
         .toPandas() \
         .to_csv(f"{TAXI_STREAM_DIR}/ride_{i:04d}.csv", index=False)

26/05/31 16:15:24 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [100]:
taxi_schema = StructType([
    StructField("VendorID",               IntegerType()),
    StructField("tpep_pickup_datetime",   TimestampType()),
    StructField("tpep_dropoff_datetime",  TimestampType()),
    StructField("passenger_count",        IntegerType()),
    StructField("trip_distance",          DoubleType()),
    StructField("pickup_longitude",       DoubleType()),
    StructField("pickup_latitude",        DoubleType()),
    StructField("RatecodeID",             IntegerType()),
    StructField("store_and_fwd_flag",     StringType()),
    StructField("dropoff_longitude",      DoubleType()),
    StructField("dropoff_latitude",       DoubleType()),
    StructField("payment_type",           IntegerType()),
    StructField("fare_amount",            DoubleType()),
    StructField("extra",                  DoubleType()),
    StructField("mta_tax",                DoubleType()),
    StructField("tip_amount",             DoubleType()),
    StructField("tolls_amount",           DoubleType()),
    StructField("improvement_surcharge",  DoubleType()),
    StructField("total_amount",           DoubleType()),
])

taxi_stream = (
    spark.readStream
    .option("maxFilesPerTrigger", 1)
    .option("header", True)
    .format("csv")
    .schema(taxi_schema)
    .load(TAXI_STREAM_DIR)
)

In [ ]:
def process_batch(batch_df, batch_id):
    if batch_df.count() == 0:
        return

    df = batch_df

    df = df.drop("VendorID", "store_and_fwd_flag", "payment_type",
                 "passenger_count", "total_amount", "tip_amount",
                 "mta_tax", "improvement_surcharge", "tpep_dropoff_datetime")


    df = (df.withColumn("pickup_hour", hour(col("tpep_pickup_datetime")).cast("integer"))
            .drop("tpep_pickup_datetime"))


    EARTH_RADIUS = 6371.0
    lat1, lon1 = radians(col("pickup_latitude")),  radians(col("pickup_longitude"))
    lat2, lon2 = radians(col("dropoff_latitude")), radians(col("dropoff_longitude"))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a_expr = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    haversine_dist = EARTH_RADIUS * 2 * atan2(sqrt(a_expr), sqrt(1 - a_expr))

    df = df.withColumn("trip_distance",
            when((col("trip_distance") <= 0) | col("trip_distance").isNull(), haversine_dist)
            .otherwise(col("trip_distance")))


    pickup_area_expr  = F.lit("Other")
    dropoff_area_expr = F.lit("Other")
    for area_name, (h_lon, h_lat) in NYC_HOTSPOTS.items():
        in_pickup  = (pow(col("pickup_longitude")  - h_lon, 2) + pow(col("pickup_latitude")  - h_lat, 2)) < 0.05**2
        in_dropoff = (pow(col("dropoff_longitude") - h_lon, 2) + pow(col("dropoff_latitude") - h_lat, 2)) < 0.05**2
        pickup_area_expr  = when(in_pickup,  F.lit(area_name)).otherwise(pickup_area_expr)
        dropoff_area_expr = when(in_dropoff, F.lit(area_name)).otherwise(dropoff_area_expr)

    df = (df.withColumn("area_pickup",  pickup_area_expr)
            .withColumn("area_dropoff", dropoff_area_expr)
            .drop("pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"))


    df = (df.filter(col("tolls_amount") <= 50)
            .withColumn("extra", _abs(col("extra"))))


    df = df.withColumn("is_airport_trip",
            col("RatecodeID").isin(2, 3) |
            col("area_pickup").isin("JFK_Airport", "LaGuardia_Airport") |
            col("area_dropoff").isin("JFK_Airport", "LaGuardia_Airport"))


    for col_name, params in learned_parameters.items():
        if col_name in df.columns:
            df = df.withColumn(col_name,
                    when(col("is_airport_trip"), col(col_name))
                    .when(col(col_name) > params["upper_bound"], F.lit(float(params["mean"])))
                    .otherwise(col(col_name)))


    cols_to_drop = ([c + "_index" for c in categorical_cols] +
                    [c + "_ohe"   for c in categorical_cols] +
                    ["unscaled_features", "features"])

    df = (model_rf_time.transform(df)
                       .withColumnRenamed("prediction", "predicted_time")
                       .drop(*cols_to_drop))

    df = (model_rf_fare.transform(df)
                       .withColumnRenamed("prediction", "predicted_fare"))

    for result in df.collect():
        print(f"\n[EVENT] Incoming Live Ride Request")
        print(f"Request ID: TR-{batch_id:04d}")
        print("-" * 50)
        print(f"  Route:    {result['area_pickup']} → {result['area_dropoff']}")
        print(f"  Schedule: {result['pickup_hour']:02d}:00")
        print(f"  Distance: {result['trip_distance']:.2f} km")
        print("-" * 50)
        print(f"  Est. Duration: {result['predicted_time']:.1f} sec")
        print(f"  Est. Fare:     ${result['predicted_fare']:.2f}")
        print("-" * 50)

taxi_query = (
    taxi_stream.writeStream
    .trigger(processingTime="10 seconds")
    .foreachBatch(process_batch)
    .queryName("taxi_stream")
    .start()
)

print("Stream going")

26/05/30 14:05:56 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-fdd633b4-88d3-4504-867b-4f74d2a90d92. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/30 14:05:56 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Stream going



[EVENT] Incoming Live Ride Request
Request ID: TR-0000
--------------------------------------------------
  Route:    Brooklyn_Bridge → Long_Island_City
  Schedule: 22:00
  Distance: 1.35 km
--------------------------------------------------
  Est. Duration: 472.9 sec
  Est. Fare:     $7.58
--------------------------------------------------

[EVENT] Incoming Live Ride Request
Request ID: TR-0001
--------------------------------------------------
  Route:    LaGuardia_Airport → Long_Island_City
  Schedule: 12:00
  Distance: 9.16 km
--------------------------------------------------
  Est. Duration: 1527.0 sec
  Est. Fare:     $26.88
--------------------------------------------------

[EVENT] Incoming Live Ride Request
Request ID: TR-0002
--------------------------------------------------
  Route:    Long_Island_City → Long_Island_City
  Schedule: 20:00
  Distance: 1.14 km
--------------------------------------------------
  Est. Duration: 394.0 sec
  Est. Fare:     $6.71
--------------

In [102]:
taxi_query.stop()
print("Stop")

Stop


In [4]:
spark.stop()